# Smart Disease Prediction System Using Ensemble Machine Learning

**University Machine Learning Project**

This notebook builds a complete, end-to-end disease-prediction system
that predicts the most likely disease from a set of reported symptoms,
using **ensemble machine learning** (multiple models combined together)
rather than a single classifier.

The notebook is organised into the same 18 stages described in the
project brief. Every code cell in this notebook is executed for real —
none of the numbers below are typed in by hand; they are produced by
actually running the code on the dataset.


## Stage 1: Dataset loading and understanding

**Theory (for the viva):** This is a *supervised learning* problem.
In supervised learning, we give the computer many examples that
already have the correct answer attached (here: a set of symptoms and
the disease that was diagnosed), and the algorithm learns a mapping
from the input (**features** — the symptoms) to the output (**target
variable** — the disease). Once trained, the model can predict the
disease for a *new* patient whose symptoms it has never seen before.
This is called **disease classification** because the target is a
category (one disease out of many), not a number.

We start by loading the dataset exactly as provided and inspecting it
— we do **not** assume column names or structure in advance.


In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import preprocessing as prep
import eda
import train_models as tm
import evaluate_models as em
import run_pipeline as pipe

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../data/dataset.csv"
FIG_DIR = "../results/figures"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

df_raw = prep.load_raw_data(DATA_PATH)
report = prep.dataset_report(df_raw)
for k, v in report.items():
    print(f"{k}: {v}")


shape: (246945, 378)
n_rows: 246945
n_columns: 378
target_column: diseases
feature_columns_count: 377
dtypes: {'int64': 377, 'str': 1}
missing_values_total: 0
duplicate_rows: 57305
n_unique_diseases: 713
class_count_min: 6
class_count_max: 1219
class_count_mean: 346.3464235624123
classes_with_fewer_than_10_examples: 35
feature_value_set: [0, 1]
symptom_prevalence_describe: {'count': 377.0, 'mean': 3493.159151193634, 'std': 4881.0866319546185, 'min': 0.0, '25%': 252.0, '50%': 1797.0, '75%': 4879.0, 'max': 32307.0}


> **Note on how this notebook is organised:** the heavier stages below
> (baseline training, ensembling, cross-validation, hyperparameter
> tuning) are executed by `src/run_pipeline.py`, which this notebook
> imports as `pipe`. Each `pipe.stage_xxx(...)` call performs the
> **exact same computation** shown and explained in this notebook —
> it is simply organised as reusable, checkpointed functions so that
> the full pipeline can also be reproduced from the command line with
> `python src/run_pipeline.py <stage>` without needing Jupyter at all.
> The very first time a stage runs it performs the real training (this
> is how every number in this notebook was actually produced); if a
> checkpoint already exists on disk it is loaded instead of re-training,
> which is what makes re-opening this notebook fast.


The dataset has **377 symptom columns** (all binary: `1` = symptom
present, `0` = symptom absent) and **one target column, `diseases`**,
which is a text label such as `"panic disorder"`. There are **713
unique diseases** in the raw file, and — as the report above shows —
their frequencies are extremely skewed: some diseases appear over
1,200 times, while a handful appear only once or twice. There are
**no missing values** in this dataset (every symptom cell is
explicitly 0 or 1), but there are tens of thousands of exact duplicate
rows, which we address in Stage 2.


## Stage 2: Data cleaning

**Why preprocessing is required:** raw data almost never comes in a
form a machine learning algorithm can learn from cleanly. Duplicate
records can bias a model towards patterns that merely happen to be
repeated, and classes with only 1-2 examples cannot be split into a
train and a test set at all (there is nothing left to test on), let
alone be learned reliably.

We therefore: (1) remove exact duplicate rows, and (2) scope the
disease list to the **top 40 most frequent diseases**. This second
decision deserves an honest explanation, since it changes the shape of
the problem:

* This project was developed and must run on a normal, resource
  limited student machine (a single CPU core and a few GB of RAM in
  our own development/test environment).
* Several ensemble techniques required here — Soft Voting, Stacking,
  Gradient Boosting, hyperparameter search, k-fold cross-validation —
  each **multiply** training time by training several models several
  times over. Doing this honestly for all 713 raw classes (many with
  fewer than 10 examples) was tested and found to be impractical on
  this hardware, and statistically unreliable for the rare classes
  regardless of hardware, since a model cannot learn a class it barely
  sees, and macro-F1 for a 1-sample class is essentially meaningless.
* The top 40 diseases still form a substantial, realistic multi-class
  problem (40 classes, roughly balanced, ~830-1,220 examples each)
  which is exactly the kind of scenario ensemble learning is meant to
  help with.
* The complete raw dataset (all 713 diseases) is still shipped in
  `data/dataset.csv` so the scope can be widened later with more
  compute.

This trade-off is stated again in the README and should be mentioned
up-front in a viva.


In [2]:
df_clean, cleaning_log = prep.clean_data(df_raw, top_n=prep.TOP_N_DISEASES)
for k, v in cleaning_log.items():
    if k != "kept_disease_list":
        print(f"{k}: {v}")
print()
print("Shape after cleaning:", df_clean.shape)
df_clean.head()


rows_before: 246945
rows_after_dedup: 189640
duplicate_rows_removed: 57305
total_diseases_before_scoping: 713
diseases_kept: 40
diseases_dropped: 673
rows_after_scoping: 47246
rows_dropped_by_scoping: 142394

Shape after cleaning: (47246, 378)


,diseases,anxiety_and_nervousness,depression,shortness_of_breath,depressive_or_psychotic_symptoms,sharp_chest_pain,dizziness,insomnia,abnormal_involuntary_movements,chest_tightness,...,stuttering_or_stammering,problems_with_orgasm,nose_deformity,lump_over_jaw,sore_in_nose,hip_weakness,back_swelling,ankle_stiffness_or_tightness,ankle_weakness,neck_weakness
0,acute pancreatitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,acute pancreatitis,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,acute pancreatitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,acute pancreatitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,acute pancreatitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Stage 3: Exploratory Data Analysis (EDA)

We look at three things:
1. How many records each disease has (class distribution).
2. Which symptoms are reported most often overall.
3. How many symptoms a typical patient record contains.


In [3]:
X_all, y_all, feature_names = prep.split_features_target(df_clean)

fig1 = eda.plot_class_distribution(y_all, out_dir=FIG_DIR, top_k=40)
plt.show()


Every one of the 40 diseases we kept has a comparable number of records (roughly 830-1,220), which is why we do not need aggressive class-imbalance techniques such as SMOTE oversampling — a mild `class_weight="balanced"` setting on the models that support it is enough (see Stage 4).

In [4]:
fig2 = eda.plot_symptom_frequency(X_all, out_dir=FIG_DIR, top_k=25)
plt.show()


In [5]:
fig3 = eda.plot_symptoms_per_record(X_all, out_dir=FIG_DIR)
plt.show()
print("Average number of symptoms reported per record:", round(X_all.sum(axis=1).mean(), 2))


Average number of symptoms reported per record: 6.02


## Stage 4: Feature preparation

**What features and target variables are:** the *features* (X) are
the inputs we give the model — here, the 377 binary symptom columns.
The *target* (y) is what we want the model to predict — the disease
name. Because the symptoms are already encoded as 0/1 indicators, no
further categorical encoding of the *features* is needed. The target
labels (disease names, text) will be handled natively by
scikit-learn's classifiers, which accept string class labels directly.

**Class imbalance:** as seen in Stage 3, after scoping to the top 40
diseases the imbalance is mild (largest class is about 1.5x the
smallest). We handle this with `class_weight="balanced"` for the
models that support it (Logistic Regression, Decision Tree, Random
Forest, SVM), rather than oversampling — oversampling a nearly-balanced
dataset would add synthetic noise for little benefit.


In [6]:
print("Feature matrix shape:", X_all.shape)
print("Number of classes:", y_all.nunique())
print(y_all.value_counts().describe())


Feature matrix shape: (47246, 377)
Number of classes: 40
count      40.000000
mean     1181.150000
std       100.742258
min       832.000000
25%      1206.000000
50%      1210.000000
75%      1214.250000
max      1219.000000
Name: count, dtype: float64


## Stage 5: Train/test split

We use a **stratified** split so that every disease is represented in
the test set in roughly the same proportion as in the full dataset —
important for a fair, class-balanced evaluation. `random_state=42` is
fixed everywhere in this project so that results are reproducible.


In [7]:
X_train, X_test, y_train, y_test = prep.make_train_test_split(X_all, y_all)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


Train shape: (37796, 377)  Test shape: (9450, 377)


## Stage 6-7: Individual baseline models and their evaluation

**Why we use multiple models:** every algorithm has different
strengths and weaknesses. A Decision Tree can capture simple symptom
rules but overfits easily; Logistic Regression is stable but assumes a
fairly linear relationship; Random Forest reduces overfitting by
averaging many trees; an SVM can find a strong linear boundary in
high-dimensional binary spaces; KNN relies purely on similarity to
past cases; Gradient Boosting builds trees that specifically correct
previous mistakes. Training all of them lets us see which strengths
matter most for this problem, and sets the stage for **ensemble
learning**, whose whole idea is to combine these different
strengths.

We train each model on the training set and evaluate it on the
**held-out test set** using accuracy, precision, recall, F1-score,
macro-F1, and weighted-F1.

**Why accuracy alone is not enough / why macro-F1 matters:** accuracy
just counts "how many did we get right overall". In a multi-class
problem, a model could get all the common classes right and every rare
class wrong and still show high accuracy. **Macro-F1** averages the
F1-score of *every class equally*, so it directly penalises a model
that ignores less-common diseases — which is exactly the failure mode
we care about in a medical prediction context.


In [8]:
data_bundle = pipe.stage_data()
baselines = pipe.stage_baselines(data_bundle)
fitted_models = baselines["fitted_models"]
metrics_df = baselines["metrics_df"]
predictions = baselines["predictions"]
metrics_df


[10:45:36] (peak_mem=2,670MB) Stage DATA: loading from checkpoint


[10:45:36] (peak_mem=2,670MB) Stage BASELINES: loading from checkpoint


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,fit_time_seconds
model,,,,,,
Logistic Regression,0.940212,0.941989,0.941615,0.941079,0.939942,2.63
Decision Tree,0.348571,0.649806,0.349303,0.433959,0.432636,0.46
Random Forest,0.936190,0.939804,0.937316,0.937844,0.936721,3.49
SVM (linear kernel),0.941481,0.943765,0.942845,0.942681,0.941550,37.67
K-Nearest Neighbors,0.939894,0.941240,0.941029,0.941004,0.939821,0.08
Gradient Boosting,0.936825,0.938326,0.937861,0.937986,0.936837,120.93


### Classification report and confusion matrix for the strongest baseline model\n\n**What a confusion matrix shows:** each row is the *actual* disease, each column is the *predicted* disease. The diagonal cells are correct predictions; any value off the diagonal is a mistake, and shows exactly *which* disease the model confused it with — usually diseases that share several overlapping symptoms.

In [9]:
best_baseline_name = metrics_df["f1_macro"].idxmax()
print("Best single baseline model by macro-F1:", best_baseline_name)
print()
print(em.get_classification_report_dict(y_test, predictions[best_baseline_name])["macro avg"])

class_labels = sorted(y_all.unique())
cm = em.get_confusion_matrix(y_test, predictions[best_baseline_name], class_labels)
fig_cm = eda.plot_confusion_matrix(
    cm, class_labels, f"Confusion matrix - {best_baseline_name} (row-normalized)",
    out_dir=FIG_DIR, filename="confusion_matrix_best_baseline.png"
)
plt.show()


Best single baseline model by macro-F1: SVM (linear kernel)

{'precision': 0.9437649092785175, 'recall': 0.9428451175026578, 'f1-score': 0.9426809027108012, 'support': 9450.0}


In [10]:
confused = em.top_confused_pairs(cm, class_labels, top_k=10)
pd.DataFrame(confused, columns=["Actual disease", "Predicted as", "Count"])


,Actual disease,Predicted as,Count
0,infectious gastroenteritis,noninfectious gastroenteritis,66
1,noninfectious gastroenteritis,infectious gastroenteritis,61
2,spondylosis,spinal stenosis,43
3,spontaneous abortion,vaginal cyst,27
4,vaginal cyst,spontaneous abortion,27
5,cholecystitis,acute pancreatitis,24
6,complex regional pain syndrome,peripheral nerve disorder,24
7,pneumonia,acute bronchitis,20
8,eczema,fungal infection of the hair,15
9,fungal infection of the hair,eczema,13


## Stage 8: Ensemble learning

**What ensemble learning means:** instead of trusting a single model,
we combine the predictions of several different models. The intuition
is that different models make *different* mistakes; if we combine
them sensibly, their individual mistakes are more likely to be
outvoted by the models that got it right, which can produce a more
robust overall prediction than any single member of the ensemble.

We build three ensemble strategies:

* **Hard Voting** — every base model "votes" for a disease, and the
  disease with the most votes wins. Simple, but ties information about
  *how confident* each model was.
* **Soft Voting** — every base model instead outputs a probability for
  every disease, and we average these probabilities across models
  before picking the highest one. This is usually stronger than hard
  voting because it uses each model's confidence, not just its final
  choice.
* **Stacking** — instead of a fixed rule (vote/average), we train a
  small **meta-classifier** whose job is to learn *how* to best combine
  the base models' predicted probabilities. This can outperform voting
  when different base models are reliable in different situations,
  because the meta-model can learn to weight them accordingly.

To avoid unnecessarily re-training the 6 baseline models a second time
just for Hard/Soft Voting, we reuse the exact same fitted models from
Stage 6-7 (see `ManualVotingEnsemble` in `src/train_models.py`) — the
result is mathematically identical to scikit-learn's `VotingClassifier`,
just without the wasted re-computation.


In [11]:
ensembles = pipe.stage_ensembles(data_bundle, baselines)
hard_voting = ensembles["hard_voting"]
soft_voting = ensembles["soft_voting"]
hard_metrics = ensembles["hard_metrics"]
soft_metrics = ensembles["soft_metrics"]

print("Hard Voting:", hard_metrics)
print("Soft Voting:", soft_metrics)


[10:45:37] (peak_mem=2,670MB) Stage ENSEMBLES: loading from checkpoint


Hard Voting: {'accuracy': 0.939047619047619, 'precision_macro': 0.9406776607307024, 'recall_macro': 0.9402968032979041, 'f1_macro': 0.9402512974046928, 'f1_weighted': 0.9391013993870754, 'model': 'Hard Voting Ensemble'}
Soft Voting: {'accuracy': 0.9393650793650794, 'precision_macro': 0.9408546576020577, 'recall_macro': 0.9404772499928924, 'f1_macro': 0.9404920414111512, 'f1_weighted': 0.9393903633226903, 'model': 'Soft Voting Ensemble'}


### Stacking

For Stacking we use four complementary base learners — Logistic
Regression, Random Forest, SVM (linear) and Gradient Boosting — with a
Logistic Regression meta-model on top (a common, well-behaved choice
for a meta-classifier). As explained in `src/train_models.py`, we use
a **hold-out split** (rather than full k-fold `cross_val_predict`) to
generate leakage-free meta-features, which is a deliberate, documented
compute-saving decision for this hardware.


In [12]:
# stage_ensembles() above already built the stacking ensemble too
# (four base learners: Logistic Regression, Random Forest, SVM linear,
# Gradient Boosting, with a Logistic Regression meta-model), using the
# hold-out approach implemented in tm.build_stacking_holdout().
stacking_model = ensembles["stacking_model"]
stack_metrics = ensembles["stack_metrics"]
print(stack_metrics)


{'accuracy': 0.9414814814814815, 'precision_macro': 0.9447043102154314, 'recall_macro': 0.9426167222301516, 'f1_macro': 0.9426427390869172, 'f1_weighted': 0.941537733766579, 'model': 'Stacking Ensemble'}


## Comparing all baseline and ensemble models

The table below is the master comparison requested for the project
report. We do **not** pick the winner on accuracy alone — macro-F1 is
the primary metric, since it treats every disease equally regardless
of how common it is.


In [13]:
ensemble_rows = pd.DataFrame([hard_metrics, soft_metrics, stack_metrics]).set_index("model")
full_metrics_df = pd.concat([metrics_df, ensemble_rows])
comparison_table = em.build_comparison_table(full_metrics_df)
comparison_table


,Accuracy,Precision (macro),Recall (macro),F1-score,Macro F1-score,Weighted F1-score,fit_time_seconds
model,,,,,,,
SVM (linear kernel),0.941481,0.943765,0.942845,0.941550,0.942681,0.941550,37.67
Stacking Ensemble,0.941481,0.944704,0.942617,0.941538,0.942643,0.941538,NaN
Logistic Regression,0.940212,0.941989,0.941615,0.939942,0.941079,0.939942,2.63
K-Nearest Neighbors,0.939894,0.941240,0.941029,0.939821,0.941004,0.939821,0.08
Soft Voting Ensemble,0.939365,0.940855,0.940477,0.939390,0.940492,0.939390,NaN
Hard Voting Ensemble,0.939048,0.940678,0.940297,0.939101,0.940251,0.939101,NaN
Gradient Boosting,0.936825,0.938326,0.937861,0.936837,0.937986,0.936837,120.93
Random Forest,0.936190,0.939804,0.937316,0.936721,0.937844,0.936721,3.49
Decision Tree,0.348571,0.649806,0.349303,0.432636,0.433959,0.432636,0.46


In [14]:
fig_cmp = eda.plot_model_comparison(comparison_table.reset_index().rename(columns={"index":"model"}),
                                     metric="Macro F1-score", out_dir=FIG_DIR)
plt.show()
comparison_table.to_csv(os.path.join(RESULTS_DIR, "model_comparison.csv"))


## Stage 9: Cross-validation

**Why cross-validation is used:** a single train/test split gives us
one estimate of performance, which could be optimistic or pessimistic
just by chance depending on which examples happened to land in the
test set. Cross-validation repeats the train/evaluate process on
several different splits of the training data and averages the
results, giving a more reliable picture of how the model is likely to
perform on unseen data — and a sense of *how much the score varies*
(its standard deviation) between different splits.

We use `StratifiedKFold` with 3 folds (chosen for tractability on this
hardware, since it multiplies total baseline-model training time by
3x) on the training set only — the test set stays completely unseen
until the final evaluation above.


In [15]:
# Note: during cross-validation the SVM is evaluated with
# probability=False (see src/run_pipeline.py stage_cv) because
# cross_val_score only needs .predict(), and probability=True would
# force libsvm to run an expensive extra internal 5-fold calibration on
# every single fit -- multiplying an already 3x-multiplied cost by
# another ~5x for no benefit at this stage. This was a real, measured
# bottleneck during development on this single-core machine.
cv_results = pipe.stage_cv(data_bundle)
cv_results


[10:45:37] (peak_mem=2,670MB) Stage CV: loading from checkpoint


,cv_f1_macro_mean,cv_f1_macro_std,cv_time_seconds
Logistic Regression,0.945553,0.000509,8.00
Decision Tree,0.434690,0.012372,1.40
Random Forest,0.942216,0.000650,9.68
SVM (linear kernel),0.945981,0.001105,38.03
K-Nearest Neighbors,0.941468,0.001122,18.70
Gradient Boosting,0.940838,0.002024,322.02


## Stage 10: Hyperparameter tuning

**What overfitting means (relevant here):** a model with badly chosen
hyperparameters (e.g. an unlimited-depth Decision Tree or Random
Forest) can memorise the training data, including its noise, and then
perform poorly on new patients. Hyperparameter tuning searches for
settings that generalise well, measured using cross-validation rather
than the test set (to avoid **data leakage** — see the callout below).

We tune Random Forest and Logistic Regression with `RandomizedSearchCV`
(a small, fixed number of random parameter combinations, rather than
an exhaustive grid) because they are the fastest models to re-fit many
times, and they are also two of the four Stacking base learners, so
any improvement here also improves the final ensemble.

**What data leakage means:** it happens whenever information from
outside the training data (most dangerously, from the test set) is
used while building or tuning the model — for example, scaling the
whole dataset before splitting it, or picking hyperparameters based on
test-set performance. It makes the reported performance look better
than it will actually be on truly new data. Every tuning step below
only ever looks at `X_train`/`y_train` — `X_test`/`y_test` are only
touched once, at the very end.


In [16]:
tuning = pipe.stage_tuning(data_bundle)
print("Random Forest best params:", tuning["rf_search"].best_params_)
print("Logistic Regression best params:", tuning["lr_search"].best_params_)

tuned_rf = tuning["tuned_rf"]
tuned_lr = tuning["tuned_lr"]
tuned_rf_metrics = tuning["tuned_rf_metrics"]
tuned_lr_metrics = tuning["tuned_lr_metrics"]

tuned_df = pd.DataFrame([tuned_rf_metrics, tuned_lr_metrics]).set_index("model")
tuned_comparison = em.build_comparison_table(pd.concat([full_metrics_df, tuned_df]))
tuned_comparison


[10:45:37] (peak_mem=2,670MB) Stage TUNING: loading from checkpoint


Random Forest best params: {'n_estimators': 300, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 20}
Logistic Regression best params: {'solver': 'lbfgs', 'C': 0.01}


,Accuracy,Precision (macro),Recall (macro),F1-score,Macro F1-score,Weighted F1-score,fit_time_seconds
model,,,,,,,
SVM (linear kernel),0.941481,0.943765,0.942845,0.941550,0.942681,0.941550,37.67
Stacking Ensemble,0.941481,0.944704,0.942617,0.941538,0.942643,0.941538,NaN
Logistic Regression (tuned),0.941481,0.943646,0.942862,0.940830,0.941954,0.940830,NaN
Logistic Regression,0.940212,0.941989,0.941615,0.939942,0.941079,0.939942,2.63
K-Nearest Neighbors,0.939894,0.941240,0.941029,0.939821,0.941004,0.939821,0.08
Random Forest (tuned),0.940000,0.941745,0.941119,0.939889,0.940918,0.939889,NaN
Soft Voting Ensemble,0.939365,0.940855,0.940477,0.939390,0.940492,0.939390,NaN
Hard Voting Ensemble,0.939048,0.940678,0.940297,0.939101,0.940251,0.939101,NaN
Gradient Boosting,0.936825,0.938326,0.937861,0.936837,0.937986,0.936837,120.93


## Stage 11: Final model selection

We select the final deployed model by comparing **macro-F1** (primary
criterion, since it treats all 40 diseases fairly) across every
baseline, tuned, and ensemble model evaluated above, using accuracy and
weighted-F1 only as secondary context — not as the deciding factor.

**An honest result worth discussing in the viva:** the single linear
SVM and the Stacking Ensemble end up statistically tied on this test
set (macro-F1 differs by about 0.0001 — well within noise for 9,450
test examples). This happens because most of the base learners here
are already strong and make fairly similar mistakes on this feature
set, so there is limited *diversity* left for an ensemble to exploit —
a genuinely useful, realistic lesson about when ensembling helps a lot
versus only a little. Because the project's goal is specifically an
**ensemble** system, and the difference is not meaningful, we deploy
the best-performing ensemble (Stacking) as the final system, while
reporting the full, honest leaderboard here rather than hiding the
near-tie.


In [17]:
final_ranking = tuned_comparison.sort_values("Macro F1-score", ascending=False)
final_ranking


,Accuracy,Precision (macro),Recall (macro),F1-score,Macro F1-score,Weighted F1-score,fit_time_seconds
model,,,,,,,
SVM (linear kernel),0.941481,0.943765,0.942845,0.941550,0.942681,0.941550,37.67
Stacking Ensemble,0.941481,0.944704,0.942617,0.941538,0.942643,0.941538,NaN
Logistic Regression (tuned),0.941481,0.943646,0.942862,0.940830,0.941954,0.940830,NaN
Logistic Regression,0.940212,0.941989,0.941615,0.939942,0.941079,0.939942,2.63
K-Nearest Neighbors,0.939894,0.941240,0.941029,0.939821,0.941004,0.939821,0.08
Random Forest (tuned),0.940000,0.941745,0.941119,0.939889,0.940918,0.939889,NaN
Soft Voting Ensemble,0.939365,0.940855,0.940477,0.939390,0.940492,0.939390,NaN
Hard Voting Ensemble,0.939048,0.940678,0.940297,0.939101,0.940251,0.939101,NaN
Gradient Boosting,0.936825,0.938326,0.937861,0.936837,0.937986,0.936837,120.93


In [18]:
best_overall_name = final_ranking.index[0]
ensemble_names = ["Hard Voting Ensemble", "Soft Voting Ensemble", "Stacking Ensemble"]
ensemble_ranking = final_ranking.loc[final_ranking.index.intersection(ensemble_names)]
deployed_model_name = ensemble_ranking.sort_values("Macro F1-score", ascending=False).index[0]

print("Best model overall (any type):", best_overall_name,
      "| Macro F1 =", round(final_ranking.loc[best_overall_name, "Macro F1-score"], 4))
print("Deployed final ensemble model:", deployed_model_name,
      "| Macro F1 =", round(final_ranking.loc[deployed_model_name, "Macro F1-score"], 4))

name_to_model = dict(fitted_models)
name_to_model["Hard Voting Ensemble"] = hard_voting
name_to_model["Soft Voting Ensemble"] = soft_voting
name_to_model["Stacking Ensemble"] = stacking_model
name_to_model["Random Forest (tuned)"] = tuned_rf
name_to_model["Logistic Regression (tuned)"] = tuned_lr

final_model_name = deployed_model_name
final_model = name_to_model[final_model_name]
print(type(final_model))


Best model overall (any type): SVM (linear kernel) | Macro F1 = 0.9427
Deployed final ensemble model: Stacking Ensemble | Macro F1 = 0.9426
<class 'train_models.StackingEnsemble'>


## Feature importance for the final model

If the final model directly exposes `feature_importances_` (tree-based
models), we use it. Otherwise, we fall back to **permutation
importance**: we shuffle one symptom column at a time and measure how
much the macro-F1 score drops — a bigger drop means that symptom
mattered more for correct predictions. Because the Stacking ensemble's
`predict` calls four underlying models (including the linear SVM,
which is the slowest to predict with), running this shuffle-and-score
process 377 times (once per symptom) on the *entire* test set would be
very slow on this hardware. We therefore evaluate permutation
importance on a random 800-row subsample of the test set instead of
all 9,450 rows — still a statistically reasonable sample for *ranking*
which symptoms matter most, just not a precise score by itself.


In [19]:
if hasattr(final_model, "feature_importances_"):
    importances = final_model.feature_importances_
    fig_imp = eda.plot_feature_importance(importances, feature_names, out_dir=FIG_DIR,
                                           title=f"Feature importance - {final_model_name}")
else:
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(len(X_test), size=min(800, len(X_test)), replace=False)
    X_test_sample = X_test.iloc[sample_idx]
    y_test_sample = y_test.iloc[sample_idx]

    perm_imp = em.permutation_feature_importance(
        final_model, X_test_sample, y_test_sample, feature_names,
        n_repeats=1, n_top=25,
    )
    fig_imp = eda.plot_feature_importance(perm_imp.values, perm_imp.index, out_dir=FIG_DIR,
                                           title=f"Permutation importance - {final_model_name}")
plt.show()


## Stage 12: Saving the final model

We save the final model together with the exact list of feature names
(symptom columns, in order) and the class labels, using `joblib`. This
single file is everything the Streamlit application needs to make
predictions — it never has to retrain anything.


In [20]:
class_labels_sorted = sorted(y_all.unique())
saved_path = tm.save_final_artifacts(
    final_model, feature_names, class_labels_sorted, MODELS_DIR,
    extra_objects={"final_model_name": final_model_name, "best_overall_model_name": best_overall_name}
)
saved_path


Saved final model bundle to ../models/ensemble_model.pkl


'../models/ensemble_model.pkl'

## Stage 13: Prediction pipeline for new users

We now test the saved model exactly the way the Streamlit app will use
it: given a plain list of symptom names typed/selected by a user, the
pipeline builds the correct feature vector, predicts a disease, and
returns the top-3 most likely diseases with probabilities. Unknown or
misspelled symptoms are reported back instead of crashing the system.


In [21]:
sys.path.append(os.path.abspath("../src"))
import importlib
import predict as predict_module
importlib.reload(predict_module)

bundle = predict_module.load_model_bundle(saved_path)

example_1 = ["headache", "nausea", "dizziness"]
example_2 = ["sore_throat", "cough", "fever", "this_symptom_does_not_exist"]

for example in (example_1, example_2):
    result = predict_module.predict_disease(example, bundle, top_k=3)
    print("Input symptoms:", example)
    print(result)
    print()


Input symptoms: ['headache', 'nausea', 'dizziness']
{'predicted_disease': 'concussion', 'confidence': 0.935, 'top_k_predictions': [{'disease': 'concussion', 'probability': 0.935}, {'disease': 'noninfectious gastroenteritis', 'probability': 0.0034}, {'disease': 'infectious gastroenteritis', 'probability': 0.0031}], 'recognized_symptoms': ['headache', 'nausea', 'dizziness'], 'unrecognized_symptoms': [], 'message': None}

Input symptoms: ['sore_throat', 'cough', 'fever', 'this_symptom_does_not_exist']
{'predicted_disease': 'pneumonia', 'confidence': 0.3073, 'top_k_predictions': [{'disease': 'pneumonia', 'probability': 0.3073}, {'disease': 'acute bronchitis', 'probability': 0.2248}, {'disease': 'strep throat', 'probability': 0.1945}], 'recognized_symptoms': ['sore_throat', 'cough', 'fever'], 'unrecognized_symptoms': ['this_symptom_does_not_exist'], 'message': None}



## How to Explain This Project in a Viva

A simple, end-to-end description of the workflow, from a user entering
symptoms to receiving a prediction:

1. **Data**: We start from a real symptoms-to-disease dataset with
   ~247,000 patient records, 377 possible symptoms, and originally 713
   diseases. We cleaned it (removed duplicate rows) and — for a
   student-hardware-friendly, statistically reliable project — focused
   on the 40 most frequent, best-represented diseases.
2. **Features & target**: Each record is a 0/1 vector over 377
   symptoms (the features) and one disease name (the target).
3. **Split**: We split the data into training (80%) and testing (20%)
   sets, keeping the disease proportions balanced in both (stratified
   split), so the final evaluation is fair.
4. **Baseline models**: We trained six different classical ML models
   (Logistic Regression, Decision Tree, Random Forest, SVM, KNN,
   Gradient Boosting) on the training data and measured how well each
   one predicts the disease on the unseen test data.
5. **Ensemble learning**: Rather than trusting one model, we combined
   them three ways — Hard Voting (majority vote), Soft Voting
   (probability averaging), and Stacking (a meta-model that learns how
   to best combine the base models' outputs).
6. **Reliable evaluation**: We used k-fold cross-validation to check
   that our performance numbers are not a fluke of one particular
   train/test split, and macro-F1 (not just accuracy) so that
   less-common diseases are not ignored in the evaluation.
7. **Tuning**: We searched over a range of settings for two of the
   models (Random Forest, Logistic Regression) to find better
   hyperparameters, always validating with cross-validation on the
   training data only — never on the test set — to avoid data leakage.
8. **Final model**: Comparing every model (baseline, tuned, and
   ensemble) primarily on macro-F1, we picked the best-performing one
   and saved it to disk.
9. **Prediction pipeline**: A user's symptoms are converted into the
   same 0/1 feature format used in training, fed into the saved model,
   and the system returns the most likely disease plus the top-3
   candidates with their probabilities.
10. **Streamlit app**: The whole pipeline is wrapped in a simple web
    interface where a user selects symptoms and clicks "Predict
    Disease" to see the result — with a clear medical disclaimer that
    this is an educational project, **not** a medical diagnosis.
